# Verify pinned Petrinex source samples

Stage 2 exploration, not a production pipeline. Run from a kernel using the installed
project environment. This notebook makes no network requests and does not modify raw
files. Acquire the exact snapshots in docs/evidence first; changed hashes require
new reviewed baselines. Source copyright: Government of Alberta. No source rows are
shown or redistributed. Counts cover the downloaded Alberta extract, not Kakwa.


In [ ]:
import json
import tempfile
from decimal import Decimal
from pathlib import Path

import duckdb

from gas_intel.profiling import profile_source, read_csv_payload

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository or notebooks directory"
baselines = json.loads((ROOT / "docs/evidence/known_month_baselines.json").read_text())

In [ ]:
for baseline in baselines:
    month = baseline["period"]
    source = ROOT / "data/raw" / baseline["source_sha256"] / "source.zip"
    assert source.exists(), f"Acquire pinned local snapshot for {month} first"
    profile = profile_source(source, month)
    assert profile["source_sha256"] == baseline["source_sha256"]
    assert profile["payload_sha256"] == baseline["payload_sha256"]
    assert profile["row_count"] == baseline["source_rows"]
    assert profile["gas_prod"]["row_count"] == baseline["qualifying_rows"]
    assert Decimal(profile["gas_prod"]["accepted_volume_sum_e3m3"]) == Decimal(
        baseline["volume_e3m3"]
    )
    payload, _ = read_csv_payload(source)
    # Independent SQL path cross-checks the Python streaming calculation.
    with tempfile.TemporaryDirectory() as temporary:
        csv_path = Path(temporary) / "source.csv"
        csv_path.write_bytes(payload)
        with duckdb.connect() as connection:
            actual = connection.execute(baseline["sql"], [str(csv_path)]).fetchone()
    expected = (
        baseline["source_rows"],
        baseline["qualifying_rows"],
        baseline["facilities"],
        Decimal(baseline["volume_e3m3"]),
    )
    assert actual == expected
    print(f"{month}: pinned counts and exact volumes match in Python and DuckDB")

## Interpretation and next decisions

Inspect docs/evidence/profile_2025-01.json and profile_2025-02.json for headers,
blank counts, code frequencies, member hashes and candidate-key diagnostics.
There are no explicit-zero GAS/PROD rows in these samples; absence is not physical
zero. Approve geography and dates and verify the stable cohort before Stage 3.
Source-wide baselines are not substitutes for the eventual cohort-specific baseline.
